# CO₂ plume segmentation

This notebook segments the five selected CO₂ matched-filter products.  Background statistics are calculated from every finite pixel in the full matched-filter image, as requested.  The threshold statistic is robust (median and MAD), so a relatively small number of plume or artefact pixels has less influence than with a scene-wide mean and standard deviation.

The notebook is intentionally limited to segmentation and quality control.  It does not calculate IME or flux: the existing downstream utilities contain CH₄-specific mass conversions and must not be applied to these CO₂ maps.

In [ ]:
from __future__ import annotations

import json
import math
import os
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import shapes
from scipy.ndimage import binary_closing, binary_opening, gaussian_filter, label
from shapely.geometry import shape
from skimage.morphology import disk


## Cases and segmentation settings

Set the `CASE_STUDIES_CO2_ROOT` environment variable (or replace the placeholder in the next cell) with the directory containing the case-study data. All spatial settings are in physical units and are converted to each raster's native grid. The values below are starting points, not validated CO₂ parameters. Run a case, inspect the candidate-component panel, then adjust that case alone. Set `selected_labels` to a list such as `[1, 4]` after review to retain only the components that belong to the plume; leave it as `None` to retain every area-filtered candidate.


In [ ]:
CASE_STUDIES_CO2_ROOT = Path(
    os.environ.get("CASE_STUDIES_CO2_ROOT", "/path/to/case_studies_co2")
).expanduser()

DEFAULT_SEGMENTATION = {
    # ΔX > full-scene median + threshold_sigma × (1.4826 × MAD)
    "threshold_sigma": 2.0,
    "smoothing_sigma_m": 60.0,
    "closing_radius_m": 60.0,
    "opening_radius_m": 30.0,
    "min_area_m2": 3_000.0,
    "selected_labels": None,
}

CASES = {
    "korba_tanager_2025-02-19": {
        "site": "Korba MegaPlant, India",
        "sensor": "Tanager-1",
        "mf_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_RGB.tif",
        "segmentation": {"smoothing_sigma_m": 75.0, "closing_radius_m": 75.0},
    },
    "korba_enmap_2023-11-06": {
        "site": "Korba MegaPlant, India",
        "sensor": "EnMAP",
        "mf_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_RGB.tif",
        "segmentation": {},
    },
    "matla_enmap_2023-10-05": {
        "site": "Matla, South Africa",
        "sensor": "EnMAP",
        "mf_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_RGB.tif",
        "segmentation": {},
    },
    "riyadh_pp09_enmap_2023-07-15": {
        "site": "Riyadh PP09, Saudi Arabia",
        "sensor": "EnMAP",
        "mf_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_RGB.tif",
        "segmentation": {},
    },
    "riyadh_pp10_enmap_2023-07-11": {
        "site": "Riyadh PP10, Saudi Arabia",
        "sensor": "EnMAP",
        "mf_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_RGB.tif",
        "segmentation": {},
    },
}

ACTIVE_CASE = "korba_enmap_2023-11-06"  # Change this key, then run the remaining cells.
RUN_STAMP = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")


In [ ]:
def pixel_size_m(src):
    """Mean native pixel size, converted to metres when the CRS is geographic."""
    res_x, res_y = src.res
    if src.crs and src.crs.is_geographic:
        _, latitude = src.transform * (src.width / 2, src.height / 2)
        metres_per_degree_lat = 111_132.92
        metres_per_degree_lon = (
            111_412.84 * math.cos(math.radians(latitude))
            - 93.5 * math.cos(3 * math.radians(latitude))
            + 0.118 * math.cos(5 * math.radians(latitude))
        )
        return float(np.mean([abs(res_x) * metres_per_degree_lon, abs(res_y) * metres_per_degree_lat]))
    return float(np.mean([abs(res_x), abs(res_y)]))


def read_single_band(path):
    with rasterio.open(path) as src:
        array = src.read(1, masked=True).filled(np.nan).astype(float)
        return array, src.profile.copy(), src.transform, src.crs, pixel_size_m(src)


def read_rgb(path):
    with rasterio.open(path) as src:
        array = src.read(masked=True).filled(np.nan).astype(float)
    if array.shape[0] < 3:
        raise ValueError(f"Expected at least three RGB bands in {path}, found {array.shape[0]}.")
    rgb = np.moveaxis(array[:3], 0, -1)
    finite = rgb[np.isfinite(rgb)]
    if finite.size:
        lo, hi = np.percentile(finite, (2, 98))
        if hi > lo:
            rgb = np.clip((rgb - lo) / (hi - lo), 0, 1)
    return rgb


def nan_gaussian(array, sigma_px):
    """Gaussian smoothing that does not pull nodata values into valid pixels."""
    if sigma_px <= 0:
        return array.copy()
    valid = np.isfinite(array)
    numerator = gaussian_filter(np.where(valid, array, 0.0), sigma=sigma_px)
    denominator = gaussian_filter(valid.astype(float), sigma=sigma_px)
    smoothed = np.divide(numerator, denominator, out=np.full_like(array, np.nan), where=denominator > 1e-12)
    smoothed[~valid] = np.nan
    return smoothed


def robust_background_statistics(array):
    """Full-scene median and Gaussian-equivalent MAD for finite pixels."""
    values = array[np.isfinite(array)]
    if values.size == 0:
        raise ValueError("The matched-filter raster has no finite pixels.")
    median = float(np.median(values))
    mad_sigma = float(1.4826 * np.median(np.abs(values - median)))
    if mad_sigma == 0:
        mad_sigma = float(np.std(values))
    if mad_sigma == 0:
        raise ValueError("The matched-filter raster is constant; a plume cannot be segmented.")
    return {"median": median, "mad_sigma": mad_sigma, "n_valid": int(values.size)}


def radius_to_structure(radius_m, pixel_m):
    radius_px = int(math.ceil(radius_m / pixel_m)) if radius_m > 0 else 0
    return (disk(max(1, radius_px)).astype(bool) if radius_px else None), radius_px


def segment(array, pixel_m, settings):
    smoothed = nan_gaussian(array, settings["smoothing_sigma_m"] / pixel_m)
    stats = robust_background_statistics(smoothed)
    threshold = stats["median"] + settings["threshold_sigma"] * stats["mad_sigma"]
    candidate = np.isfinite(smoothed) & (smoothed > threshold)

    close_structure, closing_radius_px = radius_to_structure(settings["closing_radius_m"], pixel_m)
    open_structure, opening_radius_px = radius_to_structure(settings["opening_radius_m"], pixel_m)
    if close_structure is not None:
        candidate = binary_closing(candidate, structure=close_structure)
    if open_structure is not None:
        candidate = binary_opening(candidate, structure=open_structure)
    candidate &= np.isfinite(array)

    labels, number_of_labels = label(candidate, structure=np.ones((3, 3), dtype=int))
    sizes = np.bincount(labels.ravel(), minlength=number_of_labels + 1)
    min_pixels = int(math.ceil(settings["min_area_m2"] / pixel_m**2))
    area_filtered = np.flatnonzero(sizes >= min_pixels)
    area_filtered = area_filtered[area_filtered != 0]

    requested = settings["selected_labels"]
    selected = area_filtered if requested is None else np.intersect1d(area_filtered, np.asarray(requested, dtype=int))
    final_mask = np.isin(labels, selected)
    return {
        "smoothed": smoothed, "threshold": float(threshold), "stats": stats,
        "candidate_mask": np.isin(labels, area_filtered), "final_mask": final_mask,
        "labels": labels, "sizes": sizes, "area_filtered_labels": area_filtered.tolist(),
        "selected_labels": selected.tolist(), "min_pixels": min_pixels,
        "smoothing_sigma_px": settings["smoothing_sigma_m"] / pixel_m,
        "closing_radius_px": closing_radius_px, "opening_radius_px": opening_radius_px,
    }


def write_mask(path, mask_array, profile):
    output_profile = profile.copy()
    output_profile.update(count=1, dtype=rasterio.uint8, nodata=0, compress="lzw")
    with rasterio.open(path, "w", **output_profile) as dst:
        dst.write(mask_array.astype(np.uint8), 1)


def component_geodataframe(labels, selected_labels, transform, crs, pixel_m):
    selected = set(selected_labels)
    records = []
    for geometry, component_id in shapes(labels.astype(np.int32), mask=labels > 0, transform=transform):
        component_id = int(component_id)
        pixels = int(np.count_nonzero(labels == component_id))
        records.append({
            "component_id": component_id,
            "pixel_count": pixels,
            "area_m2": pixels * pixel_m**2,
            "selected": component_id in selected,
            "geometry": shape(geometry),
        })
    if not records:
        return gpd.GeoDataFrame(
            {"component_id": [], "pixel_count": [], "area_m2": [], "selected": []},
            geometry=[],
            crs=crs,
        )
    return gpd.GeoDataFrame(records, crs=crs)


def plot_quality_control(case_id, case, mf, uncertainty, rgb, result, output_path):
    mf_values = mf[np.isfinite(mf)]
    unc_values = uncertainty[np.isfinite(uncertainty)]
    mf_vmin, mf_vmax = np.percentile(mf_values, (2, 99.5))
    unc_vmin, unc_vmax = np.percentile(unc_values, (2, 99))
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))

    im = axes[0, 0].imshow(mf, cmap="viridis", vmin=mf_vmin, vmax=mf_vmax)
    axes[0, 0].set_title("CO₂ matched-filter enhancement")
    fig.colorbar(im, ax=axes[0, 0], shrink=0.8, label="ΔX (ppm m)")

    im = axes[0, 1].imshow(uncertainty, cmap="magma", vmin=unc_vmin, vmax=unc_vmax)
    axes[0, 1].set_title("Instrument uncertainty (σ_RMN)")
    fig.colorbar(im, ax=axes[0, 1], shrink=0.8, label="σ (ppm m)")

    axes[1, 0].imshow(rgb)
    axes[1, 0].set_title("RGB quicklook")

    im = axes[1, 1].imshow(mf, cmap="viridis", vmin=mf_vmin, vmax=mf_vmax)
    axes[1, 1].contour(result["candidate_mask"], levels=[0.5], colors="white", linewidths=0.8)
    axes[1, 1].contour(result["final_mask"], levels=[0.5], colors="red", linewidths=1.5)
    axes[1, 1].set_title("Candidates (white) and final mask (red)")
    fig.colorbar(im, ax=axes[1, 1], shrink=0.8, label="ΔX (ppm m)")

    for axis in axes.ravel():
        axis.set_axis_off()
    fig.suptitle(f"{case_id} — {case['site']} — {case['sensor']}", fontsize=15)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## Run one case

The run creates `plume_segmentation_co2/<timestamp>/` next to the matched-filter raster. It writes both the area-filtered candidate mask and the final reviewed mask, component polygons, a final-plume GeoJSON, a QA figure, and a JSON manifest. The white contour in the QA figure is the full candidate set; the red contour is the final selection.

In [ ]:
if ACTIVE_CASE not in CASES:
    raise KeyError(f"Unknown case {ACTIVE_CASE!r}. Available cases: {list(CASES)}")

case = deepcopy(CASES[ACTIVE_CASE])
settings = {**DEFAULT_SEGMENTATION, **case.pop("segmentation")}
for name in ("mf_path", "uncertainty_path", "rgb_path"):
    if not case[name].is_file():
        raise FileNotFoundError(f"Missing {name}: {case[name]}")

mf, profile, transform, crs, pixel_m = read_single_band(case["mf_path"])
uncertainty, _, uncertainty_transform, uncertainty_crs, _ = read_single_band(case["uncertainty_path"])
if uncertainty.shape != mf.shape or uncertainty_transform != transform or uncertainty_crs != crs:
    raise ValueError("The uncertainty raster is not aligned with the matched-filter raster.")
rgb = read_rgb(case["rgb_path"])

result = segment(mf, pixel_m, settings)
output_dir = case["mf_path"].parent / "plume_segmentation_co2" / RUN_STAMP
output_dir.mkdir(parents=True, exist_ok=True)
stem = case["mf_path"].stem
candidate_mask_path = output_dir / f"{stem}_candidate_mask.tif"
final_mask_path = output_dir / f"{stem}_plume_mask.tif"
components_path = output_dir / f"{stem}_components.geojson"
final_polygons_path = output_dir / f"{stem}_plumes.geojson"
figure_path = output_dir / f"{stem}_segmentation_qa.png"
manifest_path = output_dir / f"{stem}_segmentation_manifest.json"

write_mask(candidate_mask_path, result["candidate_mask"], profile)
write_mask(final_mask_path, result["final_mask"], profile)
components = component_geodataframe(result["labels"], result["selected_labels"], transform, crs, pixel_m)
if not components.empty:
    components.to_file(components_path, driver="GeoJSON")
    final_polygons = components.loc[components["selected"]].copy()
    if not final_polygons.empty:
        final_polygons.to_file(final_polygons_path, driver="GeoJSON")
plot_quality_control(ACTIVE_CASE, case, mf, uncertainty, rgb, result, figure_path)

manifest = {
    "case_id": ACTIVE_CASE, "site": case["site"], "sensor": case["sensor"],
    "mf_path": str(case["mf_path"]), "uncertainty_path": str(case["uncertainty_path"]), "rgb_path": str(case["rgb_path"]),
    "pixel_size_m": pixel_m, "settings": settings, "background_statistics": result["stats"],
    "threshold_ppm_m": result["threshold"], "min_pixels": result["min_pixels"],
    "smoothing_sigma_px": result["smoothing_sigma_px"], "closing_radius_px": result["closing_radius_px"],
    "opening_radius_px": result["opening_radius_px"], "area_filtered_labels": result["area_filtered_labels"],
    "selected_labels": result["selected_labels"], "candidate_pixels": int(result["candidate_mask"].sum()),
    "final_pixels": int(result["final_mask"].sum()), "final_area_m2": float(result["final_mask"].sum() * pixel_m**2),
    "outputs": {
        "candidate_mask": str(candidate_mask_path), "final_mask": str(final_mask_path),
        "components": str(components_path) if not components.empty else None,
        "final_polygons": str(final_polygons_path) if 'final_polygons' in locals() and not final_polygons.empty else None,
        "quality_control_figure": str(figure_path),
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Full-scene background median: {result['stats']['median']:.3f} ppm m")
print(f"Full-scene robust σ (MAD): {result['stats']['mad_sigma']:.3f} ppm m")
print(f"Threshold: {result['threshold']:.3f} ppm m")
print(f"Area-filtered labels: {result['area_filtered_labels']}")
print(f"Final labels: {result['selected_labels']}")
print(f"Final area: {manifest['final_area_m2']:,.0f} m²")
print(f"Outputs: {output_dir}")


## Review loop

For each scene: run once with `selected_labels = None`; inspect the saved QA figure and `*_components.geojson`; set that case's `selected_labels` to the plume component IDs you accept; rerun with a new timestamp. The manifest preserves the full-scene statistics and exact settings used for every exported mask.